<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/loop_eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langgraph langchain langchain-openai

In [ ]:
import os
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
class State(TypedDict):
    task: str
    output: str
    feedback: str
    passed: bool
    attempts: int
    tool_result: str
    tool_used: bool
    traces: List[str]

In [ ]:
def mock_mcp_tool(query: str):
    knowledge = {
        "webhook": "A webhook is an event notification sent by another app to your server.",
        "cron": "A cron job runs a task automatically on a schedule.",
        "mcp": "MCP is a protocol that lets agents access external tools and data sources.",
        "kv cache": "KV cache stores previous key/value tensors during decoding."
    }

    for key, value in knowledge.items():
        if key in query.lower():
            return value

    return "No relevant external context found."

In [ ]:
def tool_node(state: State):
    result = mock_mcp_tool(state["task"])

    traces = state["traces"]
    traces.append(f"Tool called. Tool result: {result}")

    return {
        "tool_result": result,
        "tool_used": True,
        "traces": traces
    }

In [ ]:
def should_use_tool(state: State):
    if state["tool_used"]:
        return "verifier"

    if any(word in state["task"].lower() for word in ["webhook", "cron", "mcp", "kv cache"]):
        return "tool"

    return "verifier"

In [ ]:
def verifier_node(state: State):
    output = state["output"]
    feedback = []

    if len(output) < 80:
        feedback.append("Answer is too short.")

    if "example" not in output.lower():
        feedback.append("Answer must include an example.")

    passed = len(feedback) == 0
    feedback_text = "Passed" if passed else "\n".join(feedback)

    traces = state["traces"]
    traces.append(f"Verifier: {feedback_text}")

    return {
        "passed": passed,
        "feedback": feedback_text,
        "traces": traces
    }

In [ ]:
def verification_router(state: State):
    if state["passed"]:
        return "end"

    if state["attempts"] >= 3:
        return "end"

    return "retry"

In [ ]:
def agent_node(state: State):
    prompt = f"""
You are a clear technical teacher.

Task:
{state["task"]}

Tool context:
{state["tool_result"]}

Previous feedback:
{state["feedback"]}

Write a clear answer with:
- simple explanation
- one example
- max 150 words
"""

    response = llm.invoke(prompt)

    traces = state["traces"]
    traces.append(f"Attempt {state['attempts'] + 1}: agent generated answer.")

    return {
        "output": response.content,
        "attempts": state["attempts"] + 1,
        "traces": traces
    }

In [ ]:
graph = StateGraph(State)

graph.add_node("agent", agent_node)
graph.add_node("tool", tool_node)
graph.add_node("verifier", verifier_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_use_tool,
    {
        "tool": "tool",
        "verifier": "verifier"
    }
)

graph.add_edge("tool", "agent")

graph.add_conditional_edges(
    "verifier",
    verification_router,
    {
        "retry": "agent",
        "end": END
    }
)

app = graph.compile()

In [ ]:
def on_event(event: dict):
    print("Event triggered from:", event["source"])

    return app.invoke({
        "task": event["task"],
        "output": "",
        "feedback": "",
        "passed": False,
        "attempts": 0,
        "tool_result": "",
        "tool_used": False,
        "traces": []
    })

In [ ]:
fake_event = {
    "source": "fake_webhook",
    "task": "Explain the difference between webhook and MCP."
}

result = on_event(fake_event)

print(result["output"])

print("\nTRACES:")
for trace in result["traces"]:
    print("-", trace)

Event triggered from: fake_webhook
A webhook is a way for one application to send real-time data to another application when a specific event occurs. It's like a digital notification system that automatically sends information to your server without you having to request it.

On the other hand, MCP (Master Control Program) is a term often used in computing to refer to a central system that manages and controls various processes and resources within a network or system. It acts as a coordinator, ensuring everything runs smoothly.

**Example:** Imagine a pizza delivery service. A webhook is like the automatic text message you receive when your pizza is out for delivery. The MCP is like the manager in the kitchen who oversees the entire pizza-making process, ensuring each step is completed correctly and on time. 

In summary, webhooks are about sending notifications, while MCPs are about managing and controlling processes.

TRACES:
- Attempt 1: agent generated answer.
- Tool called. Tool 

In [ ]:
def trace_loop(traces):
    prompt = f"""
You are improving an AI agent system.

Here are the traces from one run:
{traces}

Analyse:
1. What worked?
2. What failed?
3. Should the agent prompt be improved?
4. Should the verifier be improved?
5. Give one concrete improvement for the next version.
"""

    response = llm.invoke(prompt)
    return response.content

In [ ]:
trace_result = trace_loop(result["traces"])

print("\nTRACE ANALYSIS:\n")
print(trace_result)


TRACE ANALYSIS:

Based on the provided traces, here's an analysis of the AI agent system's performance:

1. **What worked?**
   - The agent successfully generated an answer on its first attempt, indicating that it was able to process the input and produce a response.
   - The tool was effectively called, and it returned a relevant result ("A webhook is an event notification sent by another app to your server"), which suggests that the tool integration is functioning correctly.
   - The verifier passed the final output, indicating that the system's checks for correctness or relevance were satisfied.

2. **What failed?**
   - The need for a second attempt by the agent suggests that the first generated answer might not have been satisfactory or complete, prompting another attempt.
   - There is no explicit indication of what was lacking in the first attempt, which could be a point of concern if the agent frequently requires multiple attempts to generate an acceptable answer.

3. **Should